# Cleaning testing for silver layer

In [0]:
# slå ihop till en table 
@dp.table(name="marathos.silver.obt", ...)
def silver_obt():
    df_historic = spark.sql("FROM STREAM marathos.bronze.um_races_raw")
    df_new = spark.sql("FROM STREAM marathos.bronze.new_race_data")
    
    # Slå ihop till en tabell
    df = df_historic.union(df_new)
    
    # Kör samma cleaning på båda
    return clean_races(df)

In [0]:
df = spark.read.table("marathos.bronze.um_races_raw")

In [0]:
from pyspark.sql.functions import col


### Rename columns
Rename all columns to snake_case

In [0]:
import re

# Function for snackecase
def to_snakecase(text):
    return re.sub(r"[\s/]+", "_", text.strip().casefold()) # <- casefold is better than lower in this case

# Use for all columns
def rename_columns_to_snackecase(df):
    new_columns = [to_snakecase(col) for col in df.columns]
    return df.toDF(*new_columns)

df = rename_columns_to_snackecase(df)

# Verify
print(df.columns)

## Invalid rows to drop
- `Event distance/length` unit is `d` (days), `m`, `x`or unit is missing
- `Event distance/length` is `None` (missing as string): 1,053 rows
- `Event number of finishers`: drop 10 rows for "Ultra trail Dinarides - King's Race (CRO)" — all rows show 0 finishers, likely cancelled or incorrectly registered event

In [0]:
# Remove missing or invalid units

df = df.filter(
    col("event_distance_length").isNotNull() &     # nulls
    ~col("event_distance_length").rlike("d$") &    # unit is days(d)
    ~(col("event_distance_length") == "m") &     # unit is m
    ~col("event_distance_length").rlike("x$") &    # unit is x
    ~col("event_distance_length").rlike("^[0-9:.]+$") # no unit in string
)    

In [0]:
# Verify - should be 0

df.filter(
    col("event_distance_length").isNull() |     
    col("event_distance_length").rlike("d$") |   
    (col("event_distance_length") == "m") |   
    col("event_distance_length").rlike("x$") |   
    col("event_distance_length").rlike("^[0-9:.]+$") 
).count()

In [0]:
df.filter(col("event_name") == "Ultra trail Dinarides - King's Race (CRO)").display()

In [0]:
df.filter(col("event_number_of_finishers") == 0) \
    .groupBy("event_name") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(20)

In [0]:
df.count()

In [0]:
df.display(10)


## Transformations

- `Athlete performance`: string in two formats depending on event type.
Split value and unit, then convert:
  - time-based (`8:13:16 h`) → convert to seconds (integer)
  - distance-based (`60.375 km`) → extract float value




In [0]:
from pyspark.sql.functions import regexp_extract, col, when, split

# Extract — "h" or "km"
df = df.withColumn("performance_unit",
    regexp_extract(col("athlete_performance"), r"(h|km)$", 1)
)

# Extract numeric value
df = df.withColumn("performance_value",
    regexp_extract(col("athlete_performance"), r"^([\d:.]+)", 1)
)

# Time-based: "8:13:16 h" → convert to seconds
df = df.withColumn("performance_seconds",
    when(
        col("performance_unit") == "h",
        split(col("performance_value"), ":")[0].cast("double") * 3600 +
        split(col("performance_value"), ":")[1].cast("double") * 60 +
        split(col("performance_value"), ":")[2].cast("double")
    ).otherwise(None)
)

# Distance-based: "60.375 km" → extract float
df = df.withColumn("performance_km",
    when(
        col("performance_unit") == "km",
        col("performance_value").cast("double")
    ).otherwise(None)
)



- `Athlete average speed`: convert m/h values to km/h (`/ 1000`) for rows where 
  value > 500



In [0]:
from pyspark.sql.functions import col, when

df = df.withColumn("athlete_average_speed",
    when(
        col("athlete_average_speed").cast("double") > 500,
        col("athlete_average_speed").cast("double") / 1000
    ).otherwise(col("athlete_average_speed").cast("double"))
)

- `Athlete average speed` (time-based events): rename to `athlete_elapsed_time` for `h`-events, 
  then calculate correct `athlete_average_speed` = `performance_km / elapsed_hours` 

In [0]:
# Save elapsed time in new column
df = df.withColumn("athlete_elapsed_time",
    when(
        col("athlete_average_speed").rlike("^[0-9]+:[0-9]+:[0-9]+$"),
        col("athlete_average_speed")
    ).otherwise(None)
)

# Change to NULL for rows with elapsed time
df = df.withColumn("athlete_average_speed",
    when(
        col("athlete_average_speed").rlike("^[0-9]+:[0-9]+:[0-9]+$"),
        None
    ).otherwise(col("athlete_average_speed"))
)

In [0]:
# Konvertera HH:MM:SS → decimal hours
# ex: "14:00:00" → 14.0, "06:30:00" → 6.5
df = df.withColumn("elapsed_hours",
    when(
        col("athlete_elapsed_time").isNotNull(),
        split(col("athlete_elapsed_time"), ":")[0].cast("double") +
        split(col("athlete_elapsed_time"), ":")[1].cast("double") / 60 +
        split(col("athlete_elapsed_time"), ":")[2].cast("double") / 3600
    ).otherwise(None)
)

# Beräkna korrekt average speed = performance_km / elapsed_hours
# (bara för tidslopp där vi har elapsed time)
df = df.withColumn("athlete_average_speed",
    when(
        col("elapsed_hours").isNotNull() & col("performance_km").isNotNull(),
        col("performance_km") / col("elapsed_hours")
    ).otherwise(col("athlete_average_speed"))
)


### Type casting


- `Athlete year of birth`: double → integer
- `Athlete average speed`: string → double

In [0]:
# `Athlete year of birth`: double → integer
# `Athlete average speed`: string → double

from pyspark.sql.functions import to_timestamp,to_date

df = df.withColumn("athlete_year_of_birth", col("athlete_year_of_birth").cast("integer"))

df = df.withColumn("athlete_average_speed", col("athlete_average_speed").cast("double"))
                   


- `Event dates`: extract start date from inconsistent format (`06.01.2018` / `24.-26.03.2018`) → standardize to `YYYY-MM-DD` and cast to `date`
- `Event dates`: 505 rows with `00.00.YYYY` format → extract year only, 
  remaining rows extract start date and cast to `date` in Silver

### Null handling

- `Athlete club`: replace nulls with `"Unknown"` — do **not** drop rows
- `Athlete year of birth` / `Athlete age category`: leave nulls as-is, Spark handles them automatically in aggregations
- `Athlete country`, `Athlete gender`, `Athlete performance`: drop the 12 affected rows — negligible impact on dataset
- `Athlete country`: replace `XXX` with `"Unknown"`

### Data quality

#### Invalid rows to drop
- `Event distance/length` unit is `d` (days): 12,190 rows
- `Event distance/length` is `4x52km` (relay format): 4 rows
- `Event distance/length` is `None` (missing as string): 1,053 rows
- `Event distance/length` is `6:40` with performance `0:00:00 km`: invalid data
- `Year of event`: drop rows where year > 2023 (future events) — check count first
- `Event number of finishers`: drop 10 rows for "Ultra trail Dinarides - King's Race (CRO)" — all rows show 0 finishers, likely cancelled or incorrectly registered event

#### Standardization
- `Athlete age category`: map `F35` → `W35`
- `Event distance/length` units: standardize `Km`/`K`/`k` → `km` and `Miles`/`miles`/`Mile`/`mile` → `mi`
- `Event distance/length`: map `45.1m` → `45.1mi`, `71.5` → `71.5km`, `07:35` → `07:35h`
- `Athlete performance`: handle multi-day format `Xd HH:MM:SS h` → convert to total seconds in Silver
- `Event distance/length`: investigate multi-stage format `XXXkm/6Etappen` — decide keep or drop
- `Athlete country`: fix typos
  - `swe` → `SWE`
  - `Ned` → `NED`
  - `SVE` → `SWE`
  - `GRB` → `GBR`
  - `IRE` → `IRL`

#### Outliers

- `Athlete year of birth`: filter out unrealistic values (min: `1193`, max: `2021`) in **Silver**
- `Year of event`: filter out invalid years (min: `1798`) in **Silver**
- `Event number of finishers`: investigate rows where value is `0` — likely missing data stored as 0 in **Silver**
- `Athlete ID`: verify if `0` is a valid ID or placeholder for missing data in **Silver**

#### Validity checks
- If `event_distance_unit` = `km` or `mi` → `athlete_performance` must be in `h` format
  → drop rows where this does not match
- If `event_distance_unit` = `h` → `athlete_performance` must be in `km` format
  → drop rows where this does not match
  - Verify `Year of event` matches year extracted from `Event dates`


### New columns to create
- `event_type`: derive from `event_distance_unit` → `"distance"` (km/mi) or `"time"` (h) — required for gold views (Task 5) and validity checks
- `event_distance_value`: extract numeric value from `event_distance_length`
- `event_distance_unit`: extract unit from `event_distance_length`
- `event_id`: dense_rank() on `event_name`
- `athlete_id`: verify if existing column is sufficient or if new one needs to be created
- `result_id`: unique key per row

- **Dimensional modeling reminder (Task 4):** for time-based events (e.g. `6h`), the duration is constant for all athletes and belongs in `dim_event` — `Athlete performance` for these events represents distance covered, which belongs in `fct_results`. Keep this separation in mind when designing the dimensional model in Silver/Gold
